# 12주차 — LangGraph (1) 상태 그래프 · 분기 · 순환 (Colab판)

「최신인공지능」 2026 · 12주차 실습

| 실습 | 교시 | 내용 |
|------|------|------|
| 실습 1 | 2교시 | 첫 그래프 — 노드 2개를 직렬로 |
| 1-3절 ★★ | 2교시 | **리듀서** — 덮어쓸까, 누적할까 |
| 실습 2 ★★ | 2교시 | **조건부 엣지** — 분기 |
| 실습 3 ★★ | 3교시 | **Reflection 루프** — 순환 |
| 실습 4 | 3교시 | 그래프 시각화 + 실행 경로 |

> ✅ **이 차시는 Colab 이 실습실보다 편합니다.** Mermaid 다이어그램이 노트북에서 바로 렌더링되고,
> 외부 서비스 의존이 없습니다.
>
> ⚠️ 다만 실습 3(Reflection)은 반복 1회 = **LLM 호출 2회**입니다.
> **[런타임] > [런타임 유형 변경] > T4 GPU** 로 먼저 바꾸십시오.

## 0. 환경 준비

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Colab 환경 준비 — 매 세션 1회 실행 (재실행 안전)
# ══════════════════════════════════════════════════════════════
WEEK_MODELS   = ["chat"]
WEEK_PACKAGES = ("langchain langchain-core langchain-ollama python-dotenv "
                 "pydantic langsmith langgraph grandalf")
WEEK_SECRETS  = ["LANGSMITH_API_KEY"]

# ──────────────────────────────────────────────────────────────
import os, shutil, subprocess, sys, time, urllib.request

IN_COLAB = "google.colab" in sys.modules
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

GPU  = shutil.which("nvidia-smi") is not None and sh("nvidia-smi").returncode == 0
CHAT = os.environ.setdefault("MODEL", "gemma3:4b" if GPU else "gemma3:1b")

print(f"[1/5] 런타임   {'GPU 있음 ✅' if GPU else 'CPU 전용 ⚠️'}   →  대화 모델 {CHAT}")
if not GPU:
    print("       ⚠️ 실습 3(Reflection)은 반복마다 LLM 을 2회 부릅니다. T4 GPU 를 권합니다.")

print("[2/5] 패키지 설치 중…")
r = sh(f"{sys.executable} -m pip install -q {WEEK_PACKAGES}")
print("       ✅ 완료" if r.returncode == 0 else "       ❌ 실패\n" + r.stderr[-600:])

if shutil.which("ollama") is None:
    print("[3/5] Ollama 설치 중… (약 30초)")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
print("[3/5] Ollama  " + ("✅ 준비됨" if shutil.which("ollama") else "❌ 설치 실패"))

def alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not alive():
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if alive():
            break
        time.sleep(1)
print("[4/5] 서버    " + ("✅ 응답함" if alive() else "❌ 미응답 — 이 셀을 다시 실행하세요"))

have = {ln.split()[0] for ln in sh("ollama list").stdout.splitlines()[1:] if ln.strip()}
if CHAT in have:
    print(f"[5/5] {CHAT:<20s} ✅ 이미 있음")
else:
    print(f"[5/5] {CHAT:<20s} ⏳ 내려받는 중…")
    t0 = time.time()
    r = sh(f"ollama pull {CHAT}")
    print(f"       {'✅ 완료' if r.returncode == 0 else '❌ 실패'}  ({time.time() - t0:.0f}초)")

for k in WEEK_SECRETS:
    if not os.getenv(k) and IN_COLAB:
        try:
            from google.colab import userdata
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
os.environ.setdefault("LANGSMITH_PROJECT", "week12-graph")

print("\n" + "=" * 62)
print(f"준비 완료 — MODEL='{CHAT}'")
print("=" * 62)

## 실습 1 (2교시) — 첫 그래프: 노드 2개를 직렬로

```
       ┌──────────── State (공용 상태) ────────────┐
       │  {"question": ..., "answer": ..., ...}    │
       └───────────▲──────────────▲────────────────┘
                   │ 읽고 쓴다     │
             ┌─────┴────┐   ┌─────┴────┐
START ──Edge──▶│  Node A  │──▶│  Node B  │──Edge──▶ END
             └──────────┘   └──────────┘
```

| 부품 | 정체 | 한 줄 정의 |
|---|---|---|
| **State** | `TypedDict` | 노드들이 **함께 읽고 쓰는 공용 저장소** ★ |
| **Node** | **함수** | 상태를 받아 **갱신할 부분만** 돌려준다 |
| **Edge** | 연결 | **다음에 어디로 갈지** |

> ### ★ 체인과의 결정적 차이는 State 입니다
>
> 체인은 값이 **통과**하지만, 그래프는 값이 **머물러 있고** 노드들이 그것을 고칩니다.
> 그래서 **되돌아가도 그동안의 작업이 남아 있습니다.**
>
> ⚖️ 솔직하게 말하면 — **직렬 처리만 할 거면 LCEL 이 더 짧고 읽기 쉽습니다.**
> 그래프는 **분기와 순환이 필요할 때** 값을 합니다. 다음 실습부터가 본론입니다. ★

In [ ]:
import os
from typing import TypedDict

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph

MODEL = os.environ["MODEL"]
llm = ChatOllama(model=MODEL, temperature=0)


# ── ① 상태 스키마 ───────────────────────────────────────────
class State(TypedDict):
    question: str
    keywords: str
    answer: str


# ── ② 노드 = 함수 ★ ─────────────────────────────────────────
def extract_keywords(state: State) -> dict:
    p = ChatPromptTemplate.from_template("다음 질문의 핵심 키워드 3개만 쉼표로: {q}")
    kw = (p | llm | StrOutputParser()).invoke({"q": state["question"]})
    print(f"  [노드1] 키워드 = {kw.strip()[:60]}")
    return {"keywords": kw.strip()}     # ★ 갱신할 키만 — question 은 그대로 유지된다


def write_answer(state: State) -> dict:
    p = ChatPromptTemplate.from_template(
        "질문: {q}\n핵심 키워드: {kw}\n\n키워드를 반영해 세 문장으로 답하라.")
    text = (p | llm | StrOutputParser()).invoke(
        {"q": state["question"], "kw": state["keywords"]})
    print(f"  [노드2] 답변 작성 완료 ({len(text)}자)")
    return {"answer": text}


# ── ③ 조립 ──────────────────────────────────────────────────
builder = StateGraph(State)
builder.add_node("extract", extract_keywords)      # ① 노드 등록
builder.add_node("write", write_answer)

builder.add_edge(START, "extract")                 # ② 엣지 연결
builder.add_edge("extract", "write")
builder.add_edge("write", END)

first_graph = builder.compile()                    # ③ 컴파일 → Runnable 이 된다 ★

# ── ④ 실행 ─────────────────────────────────────────────
out = first_graph.invoke({"question": "LangGraph를 왜 쓰나요?"})

print("\n최종 상태 키:", list(out.keys()))          # ★ 전부 남아 있다
print("-" * 60)
print(out["answer"].strip())
print("-" * 60)

# ★ compile() 의 결과도 Runnable 입니다 — batch/stream 을 그대로 씁니다
print("\ngraph 의 타입:", type(first_graph).__name__)
print("invoke / batch / stream 을 갖고 있는가:",
      all(hasattr(first_graph, m) for m in ("invoke", "batch", "stream")))

### 관찰 포인트 ★

| 관찰 | 의미 |
|---|---|
| 최종 결과에 `question`·`keywords`·`answer` 가 **모두 있다** | *"체인이라면 앞의 값은 사라졌습니다"* ★ |
| 노드가 **갱신할 키만** 반환 | 나머지는 LangGraph 가 알아서 합칩니다 (5주차 `assign` 과 비슷한 발상) |
| `compile()` 의 결과도 Runnable | 5주차 규약이 그래프에도 적용 → **체인 안에 그래프를 끼울 수도** ★ (14주차 서브그래프) |

> ⚠️ **이 정도면 체인이 더 간단합니다. 맞습니다.**
> **분기·순환이 없으면 LCEL 을 쓰십시오.** ★
> **다음 실습(조건부 엣지)부터가 그래프를 쓰는 이유입니다.**

## 2교시 1-3절 ★★ — 리듀서: 덮어쓸까, 누적할까

**핵심 질문: 두 노드가 같은 키를 갱신하면 어떻게 됩니까?**

```
[리듀서 없음 — 기본 동작은 덮어쓰기]
   A 실행 후: {"messages": ["A가 한 말"]}
   B 실행 후: {"messages": ["B가 한 말"]}     ⚠️ A의 말이 사라졌다!

[add_messages 를 붙이면 — 누적]
   A 실행 후: {"messages": ["A가 한 말"]}
   B 실행 후: {"messages": ["A가 한 말", "B가 한 말"]}    ✅ 쌓인다
```

> ### ⚠️⚠️ 초심자가 가장 많이 겪는 함정입니다
>
> *"대화 이력이 매번 지워지는데 원인을 모르겠다"* 의 정체가 **리듀서 누락**입니다.
> **13주차에서는 이 누락이 '무한 루프' 라는 형태로 드러납니다.** ★★

| 필드 | 리듀서 | 동작 | 언제 |
|---|---|---|---|
| `question` | 없음 | 덮어쓰기 | 값 하나만 유지하면 될 때 |
| `messages` | **`add_messages`** ★ | **누적** | 대화 이력 |
| `count` | `operator.add` | 더하기 | 재시도 횟수 세기 ★ |

In [ ]:
import operator
from typing import Annotated

from langgraph.graph.message import add_messages


# ── ① 리듀서 없음 — 덮어쓰기 ⚠️ ──────────────────────────────
class StateNoReducer(TypedDict):
    messages: list
    count: int


# ── ② 리듀서 있음 — 누적 ★ ──────────────────────────────────
class StateWithReducer(TypedDict):
    messages: Annotated[list, add_messages]     # ★ 누적한다
    count:    Annotated[int, operator.add]      # ★ 더한다
    question: str                               # 리듀서 없음 → 덮어쓰기


def node_a(state) -> dict:
    return {"messages": ["A가 한 말"], "count": 1}


def node_b(state) -> dict:
    return {"messages": ["B가 한 말"], "count": 1}


def build_reducer_graph(state_cls):
    b = StateGraph(state_cls)
    b.add_node("A", node_a)
    b.add_node("B", node_b)
    b.add_edge(START, "A")
    b.add_edge("A", "B")
    b.add_edge("B", END)
    return b.compile()


def show(label: str, out: dict) -> None:
    print(f"  [{label}]")
    print(f"    count    = {out.get('count')}")
    msgs = out.get("messages", [])
    print(f"    messages = {len(msgs)}개")
    for m in msgs:
        # add_messages 는 문자열을 HumanMessage 로 바꿔 줍니다 ★
        print(f"      · {getattr(m, 'content', m)}   ({type(m).__name__})")
    print()


print("같은 그래프(A → B), State 정의만 다릅니다.\n")

print("── ① 리듀서 없음 ⚠️ ────────────────────────────────")
show("덮어쓰기", build_reducer_graph(StateNoReducer).invoke({"messages": [], "count": 0}))

print("── ② add_messages / operator.add ★ ─────────────────")
show("누적", build_reducer_graph(StateWithReducer).invoke(
    {"messages": [], "count": 0, "question": "안녕"}))

### 읽어낼 것 ★★

**① 리듀서 없음** — B 가 A 를 **덮어썼습니다.** A 의 말이 사라졌습니다.
`count` 도 1 입니다 (1 + 1 이 아니라 **나중 값으로 교체**)

**② 리듀서 있음** — `messages` 는 2개로 **쌓였고**, `count` 는 2 로 **더해졌습니다.**

> ★ `add_messages` 는 그냥 `append` 가 아닙니다.
> **메시지 id 가 같으면 덮어쓰고, 없으면 추가**합니다.
> 그래서 상태 수정·되감기(**13주차 Time Travel**)가 가능합니다.
>
> ★ **문자열을 넣었는데 `HumanMessage` 로 바뀐 것**에 주목하십시오.
> `add_messages` 는 '메시지 리스트' 를 다루는 **전용 리듀서**입니다.
>
> ⚠️ **리듀서는 필드마다 정합니다.** 위 ②에서 `question` 은 리듀서가 없으므로 여전히 덮어쓰기입니다.
> **"전부 누적"이 아니라 "무엇을 누적할지 설계하는 것"** 입니다. ★
>
> 💡 **13주차 예고**: 에이전트에서 `add_messages` 를 빼면
> 도구 결과가 사라져 모델이 **같은 도구를 계속 요청 = 무한 루프**가 됩니다. ★★

## 실습 2 ★★ (2교시) — 조건부 엣지: 분기

```
              ┌─────────────┐
START ───────▶│  classify   │
              └──────┬──────┘
                     │  ★ 라우팅 함수가 '문자열' 을 돌려준다
        ┌────────────┼────────────┐
 "math" │      "rag" │    "chat"  │
        ▼            ▼            ▼
  ┌──────────┐ ┌──────────┐ ┌──────────┐
  │ calc     │ │ search   │ │ chat     │
  └────┬─────┘ └────┬─────┘ └────┬─────┘
       └────────────┼────────────┘
                    ▼
                   END
```

> ★ **라우팅 함수는 노드가 아닙니다.**
> 상태를 **읽기만** 하고 **다음 목적지 이름**을 돌려줍니다.
> **여기서 LLM 을 부르지 마십시오** — 분류는 앞 노드(`classify`)가 이미 했습니다.
>
> ★ **`Literal` 타입으로 분류 결과를 제한한 것**에 주목하십시오.
> 5주차 구조화 출력이 여기서 값을 합니다 — 모델이 `"수학문제"` 같은 엉뚱한 문자열을 돌려주면
> **라우팅 딕셔너리에 없어서 그래프가 죽습니다.** 스키마가 그걸 막습니다. ★

> 📌 **과제 5가 정확히 이 실습의 확장입니다.** 오늘 코드를 잘 남겨 두십시오.
> 그리고 **미니 프로젝트의 골격**으로 쓰십시오 (13주차 중간 점검에서 확인합니다). ★

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

ROUTES = ("math", "rag", "chat")


class BranchState(TypedDict):
    question: str
    category: str
    answer: str


# ── 분류 노드 — 5주차 구조화 출력을 쓴다 ★ ───────────────────
class Category(BaseModel):
    """질문 유형."""

    category: Literal["math", "rag", "chat"] = Field(
        description="계산이 필요하면 math, 문서 검색이 필요하면 rag, 그 외 잡담은 chat")


def classify(state: BranchState) -> dict:
    p = ChatPromptTemplate.from_template("질문을 분류하라: {q}")
    try:
        r = (p | llm.with_structured_output(Category)).invoke({"q": state["question"]})
        category = r.category
    except Exception as e:      # 🔶 소형 모델은 구조화 출력에서도 흔들립니다
        print(f"  [분류] ⚠️ 실패 ({type(e).__name__}) → 기본 경로로 보냅니다")
        category = "chat"
    print(f"  [분류] {category}")
    return {"category": category}


# ── 경로별 노드 ─────────────────────────────────────────────
def calc(state: BranchState) -> dict:
    p = ChatPromptTemplate.from_template("계산 문제다. 단계적으로 풀어라: {q}")
    return {"answer": (p | llm | StrOutputParser()).invoke({"q": state["question"]})}


def search(state: BranchState) -> dict:
    # 🔶 11주차 retriever 를 붙이면 그대로 RAG 경로가 됩니다 ★
    #    docs = STORE.as_retriever(search_kwargs={"k": 3}).invoke(state["question"])
    return {"answer": "[검색 경로] 문서를 찾아 답합니다 — 11주차 retriever 연결 지점"}


def chat(state: BranchState) -> dict:
    p = ChatPromptTemplate.from_template("친근하게 한두 문장으로 답하라: {q}")
    return {"answer": (p | llm | StrOutputParser()).invoke({"q": state["question"]})}


# ── 라우팅 ★ ────────────────────────────────────────────────
def route(state: BranchState) -> str:
    """다음에 갈 노드의 '이름' 을 문자열로 돌려준다. ★

    ⚠️ 분류가 틀리면 경로 전체가 틀립니다.
       소형 모델의 분류 실패에 대비해 **기본 경로(fallback)** 를 두는 것이 안전합니다.
    """
    return state["category"] if state["category"] in ROUTES else "chat"


bb = StateGraph(BranchState)
bb.add_node("classify", classify)
bb.add_node("calc", calc)
bb.add_node("search", search)
bb.add_node("chat", chat)

bb.add_edge(START, "classify")
bb.add_conditional_edges(
    "classify",                                             # 이 노드 다음에
    route,                                                  # 이 함수로 판단해서
    {"math": "calc", "rag": "search", "chat": "chat"},      # 이 노드로 간다
)
for n in ("calc", "search", "chat"):
    bb.add_edge(n, END)

branch_graph = bb.compile()

# ── 서로 다른 경로를 타는 입력 3개 ★ ────────────────────
for q in ["357 곱하기 4891은?",
          "우리 학교 휴학 규정 알려줘",
          "오늘 기분이 좀 별로야"]:
    print(f"\nQ: {q}")
    out = branch_graph.invoke({"question": q})
    print(f"   경로: {out['category']}")
    print("A:", out["answer"].strip()[:100].replace("\n", " "))

### 관찰과 확장 ★

| 관찰 | 의미 |
|---|---|
| 입력에 따라 **다른 노드가 실행됨** | 분기가 실제로 동작 ★ |
| 실행되지 않은 경로는 **비용 0** | 11주차의 *"인사에도 검색이 도는"* 낭비 해결의 실마리 ★ |
| 라우팅 함수가 문자열을 반환 | 노드가 아니라 **판단**이다 |

> ⚠️ **분류가 틀리면 경로 전체가 틀립니다.**
> `route()` 에 **기본 경로(fallback)** 를 넣어 두었습니다 — 모델이 이상한 값을 내도 그래프가 죽지 않습니다. ★
>
> 💡 **13주차 연결**: `search` 노드에 **11주차 retriever** 를 꽂으면
> 그대로 **Agentic RAG 의 뼈대**가 됩니다. *"검색이 필요한지 스스로 판단"* 이 이것입니다. ★
>
> 📌 **과제 5 = 이 실습의 확장.** 그리고 **미니 프로젝트의 골격**으로 쓰십시오.
> ⚠️ 별도로 만들면 남은 2주에 완주가 어렵습니다.

## 실습 3 ★★ (3교시) — Reflection 루프: 순환

```
           ┌──────────────────────────┐
           ▼                          │
START ─▶ [generate] ─▶ [evaluate] ────┤ "revise"
                           │
                           └─────────▶ END   "end"
```

> ★ **LCEL 로는 아예 못 쓰던 것이 여기서는 딕셔너리 한 줄입니다.**
> 파이프(`|`)는 방향이 하나뿐이라 **되돌아가는 화살표를 쓸 문법이 없습니다.**

### ⚠️⚠️ 종료 조건은 두 겹으로 겁니다

| | 방식 | 누가 |
|---|---|---|
| ① **논리적 종료** | 상태에 `attempts` 를 두고 N회 넘으면 `END` | **우리가 설계** ★ |
| ② **안전망** | `recursion_limit` | 프레임워크의 강제 차단 |

> `recursion_limit` 만 믿으면 안 됩니다. 그건 **에러를 내며 멈추는** 장치입니다.
> **정상 종료는 ①로 설계해야** 합니다. ②는 버그가 있을 때를 대비한 것입니다.

### ★★ 5주차 Self-Consistency 와 대비

| | Self-Consistency (5주) | **Reflection (오늘)** |
|---|---|---|
| 구조 | **병렬** | **순환** ★ |
| 방식 | 여러 개를 뽑아 **투표** | 하나를 **반복해서 다듬음** |
| 구현 | `batch()` | **그래프 엣지** |
| 적합 | 답이 하나로 정해진 문제 | **정답이 없는** 문제 ★ |
| 비용 | N배 (한 번에) | 반복 횟수만큼 (점증) |

> **"기법이 실행 구조를 요구한다" 의 결정적 예시입니다.**
> 다수결은 **병렬**이 없으면 못 하고, Reflection 은 **순환**이 없으면 못 합니다.

In [ ]:
MAX_ATTEMPTS    = 3      # ① 논리적 종료 ★
RECURSION_LIMIT = 10     # ② 안전망 ★

llm_r = ChatOllama(model=MODEL, temperature=0.3)

# 🔶 **1회차 답변이 눈에 띄게 부실한 과제**를 고르십시오.
#    처음부터 잘 나오면 루프가 1회에 끝나 **의미가 사라집니다.** ★
TOPIC       = "고등학생에게 벡터 데이터베이스 설명하기"
REQUIREMENT = "정확히 3문장으로, 일상에서 볼 수 있는 구체적 예시를 하나 이상 포함해"

CALLS = {"n": 0}      # 호출 수를 세어 '비용' 을 숫자로 보여줍니다 ★


class ReflectState(TypedDict):
    topic: str
    draft: str
    critique: str
    passed: bool
    attempts: int


# ── ① 생성 노드 — 지적 사항이 있으면 반영한다 ★ ───────────────
def generate(state: ReflectState) -> dict:
    if state.get("critique") and state["critique"] != "OK":
        p = ChatPromptTemplate.from_template(
            "주제: {topic}\n\n이전 초안:\n{draft}\n\n"
            "지적 사항:\n{critique}\n\n지적 사항을 반영해 다시 작성하라. 요구사항: {req}")
        text = (p | llm_r | StrOutputParser()).invoke({
            "topic": state["topic"], "draft": state["draft"],
            "critique": state["critique"], "req": REQUIREMENT})
    else:
        p = ChatPromptTemplate.from_template("주제 '{topic}'에 대해 {req} 소개 글을 써라.")
        text = (p | llm_r | StrOutputParser()).invoke(
            {"topic": state["topic"], "req": REQUIREMENT})

    CALLS["n"] += 1
    n = state.get("attempts", 0) + 1
    print(f"\n  [생성 {n}회차] {text.strip()[:70]}...")
    return {"draft": text.strip(), "attempts": n}


# ── ② 자기 평가 노드 — 이분 판정 ★ ───────────────────────────
class Critique(BaseModel):
    """초안 평가 결과."""

    passed:   bool = Field(description="요구사항을 충족하면 true")
    critique: str  = Field(description="부족한 점 한두 문장. 충족하면 'OK'")


def evaluate(state: ReflectState) -> dict:
    """★ 이분 판정으로 설계하는 이유: 7주차 LLM 판정자에서 배운 것과 같습니다.
    "충족하는가? 예/아니오" 가 소형 모델에서 5점 척도보다 훨씬 안정적입니다.
    """
    p = ChatPromptTemplate.from_template(
        "다음 글이 '{req} 소개하는지' 평가하라.\n\n주제: {topic}\n글:\n{draft}")
    try:
        r = (p | llm_r.with_structured_output(Critique)).invoke(
            {"topic": state["topic"], "draft": state["draft"], "req": REQUIREMENT})
        passed, critique = r.passed, r.critique
    except Exception as e:      # 🔶 구조화 출력 실패 시 — 루프를 멈추지 않게 처리
        print(f"  [평가] ⚠️ 실패 ({type(e).__name__}) → 통과로 처리하고 종료합니다")
        passed, critique = True, "OK (평가 실패)"

    CALLS["n"] += 1
    print(f"  [평가 {state['attempts']}회차] {'통과' if passed else '보완'} — {critique[:60]}")
    print(f"  (LLM 호출 누계: {CALLS['n']}회)")
    return {"passed": passed, "critique": critique}


# ── ③ 종료 판단 ★★ ─────────────────────────────────────────
def should_continue(state: ReflectState) -> Literal["revise", "end"]:
    if state["passed"]:
        return "end"                            # 품질 충족
    if state["attempts"] >= MAX_ATTEMPTS:
        print(f"  ⚠️ {MAX_ATTEMPTS}회 도달 — 종료")
        return "end"                            # ★ 횟수 상한 (① 논리적 종료)
    return "revise"


# ── ④ 조립 — 여기서 순환이 생긴다 ★ ──────────────────────────
rb = StateGraph(ReflectState)
rb.add_node("generate", generate)
rb.add_node("evaluate", evaluate)

rb.add_edge(START, "generate")
rb.add_edge("generate", "evaluate")
rb.add_conditional_edges(
    "evaluate", should_continue,
    {"revise": "generate", "end": END},         # ★ "generate" 로 되돌아간다 = 순환
)

reflection_graph = rb.compile()

print(f"모델: {MODEL} / 반복 상한 {MAX_ATTEMPTS} / recursion_limit {RECURSION_LIMIT}")
print(f"요구사항: {REQUIREMENT}")

out = reflection_graph.invoke(
    {"topic": TOPIC, "attempts": 0, "critique": ""},
    config={"recursion_limit": RECURSION_LIMIT},   # ★ ② 안전망
)

print("\n" + "=" * 60)
print(f"총 {out['attempts']}회 시도 / 통과 여부: {out['passed']} / LLM 호출 {CALLS['n']}회")
print("-" * 60)
print(out["draft"])
print("=" * 60)

### 학생이 채울 표 ★

| 회차 | 초안 요약 | 평가 | 호출 수 누계 |
|---|---|---|---|
| 1 | | | 2 |
| 2 | | | 4 |
| 3 | | | 6 |

### 읽어낼 것 ★★

| 관찰 | 의미 |
|---|---|
| 2회차가 1회차보다 나아짐 | **Reflection 이 작동** ★ |
| 3회차는 별로 안 나아짐 | **수확 체감** — 반복을 늘려도 한계가 있다 ★★ |
| 호출이 회차당 2회씩 증가 | **비용은 선형 증가** ⚠️ |
| 자기 평가가 계속 "보완" | ⚠️ 평가 노드가 너무 엄격 — 종료가 안 됨 |
| 1회차에 바로 "통과" | ⚠️ 평가 노드가 너무 관대 — 루프가 무의미 |

> ### ⚖️ 자기 평가의 딜레마를 반드시 짚으십시오
>
> **같은 모델이 쓰고 같은 모델이 채점합니다.**
> **모델이 못 보는 결함은 평가에서도 못 봅니다.**
> → 7주차 *"판정자를 믿을 수 있는가"* 와 정확히 같은 문제입니다. ★★
> 실무에서는 **평가에 더 강한 모델**을 쓰거나 **규칙 기반 검사**를 섞습니다.

> ### 📌 결론 문장
> ***"반복하면 좋아집니다. 다만 무한히 좋아지지는 않고, 비용은 확실히 늘어납니다."***
>
> 🔶 1회차에 바로 통과해 버리면 `REQUIREMENT` 를 더 까다롭게 바꾸십시오.
> 루프가 1회에 끝나면 이 실습의 의미가 사라집니다. ★

## 실습 4 (3교시) — 그래프 시각화

> ★ **되돌아가는 화살표가 그림에 보입니다. 이 그림을 LCEL 로는 그릴 수 없었습니다.**
> 1교시의 *"체인은 파이프, 그래프는 회로도"* 를 여기서 회수하십시오.

> ### ⚠️ 강의안 정정 ★
>
> 강의안은 `draw_ascii()` 를 *"의존성이 가장 가볍다"* 고 소개하지만,
> 실제로는 **`grandalf` 패키지가 필요합니다** (없으면 `ImportError`).
> 추가 설치 없이 되는 것은 **`draw_mermaid()`** 쪽입니다. 🔶
>
> ★ 과제 5 의 "시각화 이미지" 는 **ASCII 캡처나 Mermaid 텍스트도 인정**합니다.

In [ ]:
GRAPHS = {
    "reflection": (reflection_graph, "순환 — 되돌아가는 화살표를 확인하십시오 ★"),
    "branch":     (branch_graph,     "분기 — 세 갈래로 나뉘는 것을 확인하십시오"),
    "first":      (first_graph,      "직렬 — 분기도 순환도 없습니다 (체인이 더 낫습니다 ⚖️)"),
}

KEY = "reflection"       # "branch" / "first" 로 바꿔 보십시오
graph, note = GRAPHS[KEY]

print(f"── {KEY} 그래프 — {note}\n")

print("── ① ASCII ─────────────────────────────────────")
try:
    print(graph.get_graph().draw_ascii())
except ImportError:
    print("""🔶 draw_ascii() 에는 grandalf 가 필요합니다.
   (설치가 안 되면 아래 Mermaid 로 충분합니다 — 과제 5에서도 인정됩니다) ★""")

print("\n── ② Mermaid (README 에 그대로 붙이십시오) ★ ────")
MERMAID = graph.get_graph().draw_mermaid()
print(MERMAID)

from pathlib import Path
Path(f"{KEY}.mmd").write_text(MERMAID, encoding="utf-8")
print(f"\n✅ 저장: {KEY}.mmd   ← 좌측 📁 파일 패널에서 내려받으십시오 (과제 5 제출물) ★")

In [ ]:
# ── ③ 노트북에서 다이어그램을 그림으로 보기 ★ ──
#    Colab 은 mermaid.ink 로 렌더링합니다 (네트워크 필요).
#    실패해도 위 ASCII / Mermaid 텍스트로 충분합니다.
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"""🔶 PNG 렌더링 실패 ({type(e).__name__}: {str(e)[:60]})
   외부 렌더링 서비스가 필요합니다. **ASCII / Mermaid 로 충분합니다.** ★
   과제 5 의 '시각화 이미지' 는 ASCII 캡처나 Mermaid 텍스트도 인정합니다.""")

### LangSmith 에서 실행 경로 보기 ★

```
Trace
 ├ Run: generate    (1회차)
 ├ Run: evaluate    (1회차)
 ├ Run: generate    (2회차)   ★ 같은 노드가 다시 나온다 = 순환의 증거
 ├ Run: evaluate    (2회차)
 └ Run: generate    (3회차)
```

| 확인할 것 | 왜 |
|---|---|
| **같은 노드가 여러 번** | **순환이 실제로 돌았다는 증거** ★ |
| 회차별 `critique` 내용 | 무엇을 지적했고 반영됐는지 |
| **총 지연·토큰** | 반복의 **비용**을 숫자로 (6주차) |
| 어느 분기를 탔는가 | 조건부 엣지의 판단 결과 |

> 📌 **6주차에 배운 화면이 그래프에서도 그대로 쓰입니다.**
> 그때 *"9주차 이후 실행 경로가 복잡해지므로 관측 도구를 먼저 익힌다"* 고 한 이유가 이것입니다. ★

## 오늘 확인할 것

- [ ] 그래프 실행 후 **상태의 모든 키가 남아 있는 것**을 확인했다 ★
- [ ] 리듀서를 **뺐을 때와 넣었을 때**를 나란히 돌려 비교했다 ★★
- [ ] 조건부 엣지로 **입력에 따라 다른 노드가 실행**되는 것을 봤다 ★★
- [ ] Reflection 루프에서 **회차별 개선과 수확 체감**을 관찰했다 ★★
- [ ] 종료 조건을 **두 겹**(`attempts` + `recursion_limit`)으로 걸었다 ★
- [ ] Mermaid 다이어그램에서 **되돌아가는 화살표**를 확인했다 ★

### 📌 과제 5

**조건부 엣지 실습(실습 2)의 확장**입니다. 그리고 **미니 프로젝트의 골격**으로 쓰십시오.
제출물의 "시각화 이미지" 는 **ASCII 캡처나 Mermaid 텍스트도 인정**합니다.

### 오늘의 한 줄

> **체인은 파이프, 그래프는 회로도.**
> 되돌아가는 화살표가 필요해지는 순간이 그래프를 쓰는 순간입니다.